In [1]:
import importlib

import pandas as pd

import attrition_analysis.data_selection as data_selection
import attrition_analysis.models_utils as models_utils

importlib.reload(models_utils)
importlib.reload(data_selection)

from attrition_analysis.data_selection import CATEGORICAL_MODEL_VARS, MIXED_MODEL_VARS
from attrition_analysis.models_utils import (
    estimators_dict,
    evaluate_thresholds_optimized_models,
    mixed_models_dict_c,
    categorical_models_dict_c,
    param_distributions_dict,
    run_cross_validation_mixed,
    run_model_comparison_mixed,
    tune_hyperparameters_top_combinations,
)

df = pd.read_csv("../../data/clean/Employee-Attrition_Clean.csv")

target = "AttritionFlag"

df

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,EducationField,EmployeeCount,EmployeeNumber,Gender,...,IncomeGroup,EducationLevel,StockOption,JobLevelGroup,SatisfactionLevel,PerformanceRatingLevel,EnvironmentSatisfactionLevel,JobInvolvementLevel,RelationshipSatisfactionLevel,WorkLifeBalanceLevel
0,41,Yes,Travel_Rarely,1102,Sales,1,Life Sciences,1,1,Female,...,Medium,Low,No Options,Junior Level,High,Medium-High,Medium-Low,Medium-High,Low,Low
1,49,No,Travel_Frequently,279,Research & Development,8,Life Sciences,1,2,Male,...,Medium,Very Low,Low,Junior Level,Medium-Low,High,Medium-High,Medium-Low,High,Medium-High
2,37,Yes,Travel_Rarely,1373,Research & Development,2,Other,1,4,Male,...,Low,Low,No Options,Entry Level,Medium-High,Medium-High,High,Medium-Low,Medium-Low,Medium-High
3,33,No,Travel_Frequently,1392,Research & Development,3,Life Sciences,1,5,Female,...,Low,High,No Options,Entry Level,Medium-High,Medium-High,High,Medium-High,Medium-High,Medium-High
4,27,No,Travel_Rarely,591,Research & Development,2,Medical,1,7,Male,...,Medium,Very Low,Low,Entry Level,Medium-Low,Medium-High,Low,Medium-High,High,Medium-High
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1465,36,No,Travel_Frequently,884,Research & Development,23,Medical,1,2061,Male,...,Low,Low,Low,Junior Level,High,Medium-High,Medium-High,High,Medium-High,Medium-High
1466,39,No,Travel_Rarely,613,Research & Development,6,Medical,1,2062,Male,...,High,Very Low,Low,Mid Level,Low,Medium-High,High,Medium-Low,Low,Medium-High
1467,27,No,Travel_Rarely,155,Research & Development,4,Life Sciences,1,2064,Male,...,High,Medium,Low,Junior Level,Medium-Low,High,Medium-Low,High,Medium-Low,Medium-High
1468,49,No,Travel_Frequently,1023,Sales,2,Medical,1,2065,Male,...,Medium,Medium,No Options,Junior Level,Medium-Low,Medium-High,High,Medium-Low,High,Medium-Low


In [2]:
all_models_dict_c = {**categorical_models_dict_c, **mixed_models_dict_c}

# Comparação dos Modelos Categóricos

In [3]:
general_comparison_c, threshold_comparison_c, confusion_results_c, trained_models_c, interpretation_results_c = run_model_comparison_mixed(
    df=df,
    models_dict=categorical_models_dict_c,
    estimators_dict=estimators_dict,
    target="AttritionFlag"
)

general_comparison_c.sort_values(by="F1-score", ascending=False)

,Variable_Set,Model,Threshold,Accuracy,Precision,Recall,F1-score,AUC,N_Numeric_Variables,N_Categorical_Variables,N_Features_After_Dummies
19,Modelo 2 — Nível Hierárquico,XGBoost Balanced,0.5,0.791,0.416,0.732,0.531,0.804,0,8,22
59,Modelo 6 — Perfil Pessoal,XGBoost Balanced,0.5,0.805,0.432,0.676,0.527,0.786,0,9,24
17,Modelo 2 — Nível Hierárquico,Gradient Boosting Balanced,0.5,0.787,0.409,0.732,0.525,0.800,0,8,22
11,Modelo 2 — Nível Hierárquico,Logistic Regression Balanced,0.5,0.776,0.394,0.732,0.512,0.815,0,8,22
57,Modelo 6 — Perfil Pessoal,Gradient Boosting Balanced,0.5,0.796,0.414,0.648,0.505,0.779,0,9,24
...,...,...,...,...,...,...,...,...,...,...,...
54,Modelo 6 — Perfil Pessoal,Random Forest,0.5,0.841,0.667,0.028,0.054,0.790,0,9,24
4,Modelo 1 — Função Profissional,Random Forest,0.5,0.839,0.000,0.000,0.000,0.768,0,8,26
64,Modelo 7 — Reduzido Conservador,Random Forest,0.5,0.839,0.000,0.000,0.000,0.758,0,7,18
44,Modelo 5 — Estabilidade e Benefícios,Random Forest,0.5,0.839,0.000,0.000,0.000,0.791,0,9,24


In [4]:
c_best_thresholds_f1 = threshold_comparison_c.loc[
    threshold_comparison_c.groupby(["Variable_Set", "Model"])["F1-score"].idxmax()
].reset_index(drop=True)

c_best_thresholds_f1.sort_values(by="F1-score", ascending=False)

,Variable_Set,Model,Threshold,Accuracy,Precision,Recall,F1-score,AUC
18,Modelo 2 — Nível Hierárquico,XGBoost,0.250,0.834,0.489,0.634,0.552,0.808
13,Modelo 2 — Nível Hierárquico,Gradient Boosting Balanced,0.550,0.816,0.454,0.690,0.547,0.800
25,Modelo 3 — Faixa Salarial,Logistic Regression Balanced,0.625,0.834,0.489,0.606,0.541,0.790
15,Modelo 2 — Nível Hierárquico,Logistic Regression Balanced,0.575,0.814,0.449,0.676,0.539,0.815
12,Modelo 2 — Nível Hierárquico,Gradient Boosting,0.275,0.832,0.483,0.606,0.537,0.801
...,...,...,...,...,...,...,...,...
21,Modelo 3 — Faixa Salarial,Decision Tree Balanced,0.375,0.689,0.301,0.704,0.422,0.738
60,Modelo 7 — Reduzido Conservador,Decision Tree,0.200,0.766,0.349,0.521,0.418,0.720
61,Modelo 7 — Reduzido Conservador,Decision Tree Balanced,0.425,0.673,0.291,0.718,0.415,0.704
30,Modelo 4 — Trajetória Organizacional,Decision Tree,0.200,0.780,0.362,0.479,0.412,0.722


In [5]:
c_best_thresholds_by_model = threshold_comparison_c.loc[
    threshold_comparison_c.groupby(["Model"])["F1-score"].idxmax()
].reset_index(drop=True)

c_best_thresholds_by_model.sort_values(by="F1-score", ascending=False)

,Variable_Set,Model,Threshold,Accuracy,Precision,Recall,F1-score,AUC
8,Modelo 2 — Nível Hierárquico,XGBoost,0.250,0.834,0.489,0.634,0.552,0.808
3,Modelo 2 — Nível Hierárquico,Gradient Boosting Balanced,0.550,0.816,0.454,0.690,0.547,0.800
5,Modelo 3 — Faixa Salarial,Logistic Regression Balanced,0.625,0.834,0.489,0.606,0.541,0.790
2,Modelo 2 — Nível Hierárquico,Gradient Boosting,0.275,0.832,0.483,0.606,0.537,0.801
9,Modelo 2 — Nível Hierárquico,XGBoost Balanced,0.500,0.791,0.416,0.732,0.531,0.804
4,Modelo 3 — Faixa Salarial,Logistic Regression,0.275,0.832,0.482,0.577,0.526,0.784
6,Modelo 2 — Nível Hierárquico,Random Forest,0.250,0.832,0.481,0.535,0.507,0.796
7,Modelo 2 — Nível Hierárquico,Random Forest Balanced,0.600,0.837,0.493,0.521,0.507,0.796
0,Modelo 2 — Nível Hierárquico,Decision Tree,0.200,0.791,0.402,0.606,0.483,0.723
1,Modelo 5 — Estabilidade e Benefícios,Decision Tree Balanced,0.600,0.816,0.439,0.507,0.471,0.758


In [6]:
c_threshold_mean = threshold_comparison_c.groupby("Variable_Set")[[
    "Accuracy", "Precision", "Recall", "F1-score", "AUC"
]].mean().sort_values(by="F1-score", ascending=False)

c_threshold_mean.sort_values(by="F1-score", ascending=False)

,Accuracy,Precision,Recall,F1-score,AUC
Variable_Set,,,,,
Modelo 3 — Faixa Salarial,0.743742,0.438521,0.522753,0.392837,0.7698
Modelo 2 — Nível Hierárquico,0.743100,0.432711,0.530895,0.386279,0.7878
Modelo 6 — Perfil Pessoal,0.752216,0.430147,0.501716,0.380068,0.7850
Modelo 1 — Função Profissional,0.746863,0.420684,0.489389,0.367100,0.7644
Modelo 5 — Estabilidade e Benefícios,0.744763,0.414942,0.504042,0.363253,0.7812
Modelo 4 — Trajetória Organizacional,0.743374,0.377458,0.477384,0.357358,0.7522
Modelo 7 — Reduzido Conservador,0.726311,0.400258,0.448884,0.307705,0.7537


In [7]:
c_threshold_with_f1_05 = threshold_comparison_c[threshold_comparison_c["F1-score"]>=0.5]

c_best_thresholds_accuracy = c_threshold_with_f1_05.loc[
    c_threshold_with_f1_05.groupby(["Model"])["Accuracy"].idxmax()
].reset_index(drop=True)

c_best_thresholds_accuracy.sort_values(by="Accuracy", ascending=False)

,Variable_Set,Model,Threshold,Accuracy,Precision,Recall,F1-score,AUC
2,Modelo 5 — Estabilidade e Benefícios,Logistic Regression,0.325,0.859,0.582,0.451,0.508,0.787
6,Modelo 2 — Nível Hierárquico,XGBoost,0.375,0.857,0.571,0.451,0.504,0.808
3,Modelo 5 — Estabilidade e Benefícios,Logistic Regression Balanced,0.650,0.846,0.521,0.535,0.528,0.793
1,Modelo 3 — Faixa Salarial,Gradient Boosting Balanced,0.650,0.839,0.500,0.507,0.503,0.773
5,Modelo 2 — Nível Hierárquico,Random Forest Balanced,0.600,0.837,0.493,0.521,0.507,0.796
7,Modelo 2 — Nível Hierárquico,XGBoost Balanced,0.650,0.837,0.494,0.549,0.520,0.804
0,Modelo 3 — Faixa Salarial,Gradient Boosting,0.275,0.834,0.487,0.535,0.510,0.785
4,Modelo 2 — Nível Hierárquico,Random Forest,0.250,0.832,0.481,0.535,0.507,0.796


In [8]:
c_best_thresholds_precision = c_threshold_with_f1_05.loc[
    c_threshold_with_f1_05.groupby(["Model"])["Precision"].idxmax()
].reset_index(drop=True)

c_best_thresholds_precision.sort_values(by="Precision", ascending=False)

,Variable_Set,Model,Threshold,Accuracy,Precision,Recall,F1-score,AUC
2,Modelo 5 — Estabilidade e Benefícios,Logistic Regression,0.325,0.859,0.582,0.451,0.508,0.787
6,Modelo 2 — Nível Hierárquico,XGBoost,0.375,0.857,0.571,0.451,0.504,0.808
3,Modelo 5 — Estabilidade e Benefícios,Logistic Regression Balanced,0.650,0.846,0.521,0.535,0.528,0.793
1,Modelo 3 — Faixa Salarial,Gradient Boosting Balanced,0.650,0.839,0.500,0.507,0.503,0.773
7,Modelo 2 — Nível Hierárquico,XGBoost Balanced,0.650,0.837,0.494,0.549,0.520,0.804
5,Modelo 2 — Nível Hierárquico,Random Forest Balanced,0.600,0.837,0.493,0.521,0.507,0.796
0,Modelo 3 — Faixa Salarial,Gradient Boosting,0.275,0.834,0.487,0.535,0.510,0.785
4,Modelo 2 — Nível Hierárquico,Random Forest,0.250,0.832,0.481,0.535,0.507,0.796


In [9]:
c_best_thresholds_recall = c_threshold_with_f1_05.loc[
    c_threshold_with_f1_05.groupby(["Model"])["Recall"].idxmax()
].reset_index(drop=True)

c_best_thresholds_recall.sort_values(by="Recall", ascending=False)

,Variable_Set,Model,Threshold,Accuracy,Precision,Recall,F1-score,AUC
1,Modelo 2 — Nível Hierárquico,Gradient Boosting Balanced,0.475,0.762,0.379,0.746,0.502,0.800
7,Modelo 2 — Nível Hierárquico,XGBoost Balanced,0.475,0.787,0.411,0.746,0.530,0.804
3,Modelo 2 — Nível Hierárquico,Logistic Regression Balanced,0.500,0.776,0.394,0.732,0.512,0.815
0,Modelo 2 — Nível Hierárquico,Gradient Boosting,0.200,0.796,0.419,0.690,0.521,0.801
2,Modelo 2 — Nível Hierárquico,Logistic Regression,0.200,0.798,0.421,0.676,0.519,0.811
6,Modelo 2 — Nível Hierárquico,XGBoost,0.200,0.798,0.418,0.648,0.508,0.808
4,Modelo 2 — Nível Hierárquico,Random Forest,0.250,0.832,0.481,0.535,0.507,0.796
5,Modelo 2 — Nível Hierárquico,Random Forest Balanced,0.600,0.837,0.493,0.521,0.507,0.796


In [10]:
c_best_thresholds_auc = c_threshold_with_f1_05.loc[
    c_threshold_with_f1_05.groupby(["Model"])["AUC"].idxmax()
].reset_index(drop=True)

c_best_thresholds_auc.sort_values(by="AUC", ascending=False)

,Variable_Set,Model,Threshold,Accuracy,Precision,Recall,F1-score,AUC
3,Modelo 2 — Nível Hierárquico,Logistic Regression Balanced,0.500,0.776,0.394,0.732,0.512,0.815
2,Modelo 2 — Nível Hierárquico,Logistic Regression,0.200,0.798,0.421,0.676,0.519,0.811
6,Modelo 2 — Nível Hierárquico,XGBoost,0.200,0.798,0.418,0.648,0.508,0.808
7,Modelo 2 — Nível Hierárquico,XGBoost Balanced,0.475,0.787,0.411,0.746,0.530,0.804
0,Modelo 2 — Nível Hierárquico,Gradient Boosting,0.200,0.796,0.419,0.690,0.521,0.801
1,Modelo 2 — Nível Hierárquico,Gradient Boosting Balanced,0.475,0.762,0.379,0.746,0.502,0.800
4,Modelo 2 — Nível Hierárquico,Random Forest,0.250,0.832,0.481,0.535,0.507,0.796
5,Modelo 2 — Nível Hierárquico,Random Forest Balanced,0.600,0.837,0.493,0.521,0.507,0.796


# Comparação dos Modelos Mistos

In [11]:
general_comparison_mix, threshold_comparison_mix, confusion_results_mix, trained_models_mix, interpretation_results_mix = run_model_comparison_mixed(
    df=df,
    models_dict=mixed_models_dict_c,
    estimators_dict=estimators_dict,
    target="AttritionFlag"
)

general_comparison_mix.sort_values(by="F1-score", ascending=False)

,Variable_Set,Model,Threshold,Accuracy,Precision,Recall,F1-score,AUC,N_Numeric_Variables,N_Categorical_Variables,N_Features_After_Dummies
11,Modelo 2 — Nível Hierárquico e Benefícios,Logistic Regression Balanced,0.5,0.769,0.388,0.761,0.514,0.829,3,8,25
17,Modelo 2 — Nível Hierárquico e Benefícios,Gradient Boosting Balanced,0.5,0.810,0.436,0.620,0.512,0.824,3,8,25
19,Modelo 2 — Nível Hierárquico e Benefícios,XGBoost Balanced,0.5,0.810,0.436,0.620,0.512,0.821,3,8,25
21,Modelo 3 — Rendimento Quantitativo,Logistic Regression Balanced,0.5,0.760,0.378,0.761,0.505,0.792,4,6,19
70,Modelo 8 — Integrado Multidimensional,Logistic Regression,0.5,0.868,0.644,0.408,0.500,0.808,7,11,43
...,...,...,...,...,...,...,...,...,...,...,...
4,Modelo 1 — Função Profissional Misto,Random Forest,0.5,0.841,1.000,0.014,0.028,0.771,3,7,26
44,Modelo 5 — Antiguidade Organizacional,Random Forest,0.5,0.841,1.000,0.014,0.028,0.756,5,6,20
14,Modelo 2 — Nível Hierárquico e Benefícios,Random Forest,0.5,0.839,0.500,0.014,0.027,0.808,3,8,25
54,Modelo 6 — Perfil Pessoal e Condições de Trabalho,Random Forest,0.5,0.839,0.500,0.014,0.027,0.792,3,8,24


In [12]:
mix_best_thresholds_by_f1 = threshold_comparison_mix.loc[
    threshold_comparison_mix.groupby(["Variable_Set", "Model"])["F1-score"].idxmax()
].reset_index(drop=True)

mix_best_thresholds_by_f1.sort_values(by="F1-score", ascending=False)

,Variable_Set,Model,Threshold,Accuracy,Precision,Recall,F1-score,AUC
74,Modelo 8 — Integrado Multidimensional,Logistic Regression,0.325,0.859,0.557,0.620,0.587,0.808
75,Modelo 8 — Integrado Multidimensional,Logistic Regression Balanced,0.650,0.844,0.511,0.634,0.566,0.810
18,Modelo 2 — Nível Hierárquico e Benefícios,XGBoost,0.275,0.857,0.556,0.563,0.559,0.829
14,Modelo 2 — Nível Hierárquico e Benefícios,Logistic Regression,0.275,0.846,0.518,0.606,0.558,0.827
15,Modelo 2 — Nível Hierárquico e Benefícios,Logistic Regression Balanced,0.650,0.834,0.489,0.606,0.541,0.829
...,...,...,...,...,...,...,...,...
31,Modelo 4 — Experiência Profissional,Decision Tree Balanced,0.300,0.655,0.279,0.718,0.402,0.696
10,Modelo 2 — Nível Hierárquico e Benefícios,Decision Tree,0.200,0.680,0.287,0.662,0.400,0.712
0,Modelo 1 — Função Profissional Misto,Decision Tree,0.200,0.753,0.327,0.507,0.398,0.695
20,Modelo 3 — Rendimento Quantitativo,Decision Tree,0.200,0.755,0.327,0.493,0.393,0.735


In [13]:
mix_best_thresholds_by_model = threshold_comparison_mix.loc[
    threshold_comparison_mix.groupby(["Model"])["F1-score"].idxmax()
].reset_index(drop=True)

mix_best_thresholds_by_model.sort_values(by="F1-score", ascending=False)

,Variable_Set,Model,Threshold,Accuracy,Precision,Recall,F1-score,AUC
4,Modelo 8 — Integrado Multidimensional,Logistic Regression,0.325,0.859,0.557,0.620,0.587,0.808
5,Modelo 8 — Integrado Multidimensional,Logistic Regression Balanced,0.650,0.844,0.511,0.634,0.566,0.810
8,Modelo 2 — Nível Hierárquico e Benefícios,XGBoost,0.275,0.857,0.556,0.563,0.559,0.829
9,Modelo 2 — Nível Hierárquico e Benefícios,XGBoost Balanced,0.650,0.862,0.581,0.507,0.541,0.821
3,Modelo 2 — Nível Hierárquico e Benefícios,Gradient Boosting Balanced,0.625,0.857,0.562,0.507,0.533,0.824
7,Modelo 2 — Nível Hierárquico e Benefícios,Random Forest Balanced,0.550,0.839,0.500,0.563,0.530,0.806
2,Modelo 2 — Nível Hierárquico e Benefícios,Gradient Boosting,0.300,0.850,0.538,0.493,0.515,0.828
6,Modelo 8 — Integrado Multidimensional,Random Forest,0.200,0.789,0.408,0.690,0.513,0.787
1,Modelo 2 — Nível Hierárquico e Benefícios,Decision Tree Balanced,0.600,0.803,0.423,0.620,0.503,0.739
0,Modelo 8 — Integrado Multidimensional,Decision Tree,0.200,0.789,0.380,0.493,0.429,0.704


In [14]:
mix_threshold_mean = threshold_comparison_mix.groupby("Variable_Set")[[
    "Accuracy", "Precision", "Recall", "F1-score", "AUC"
]].mean()

mix_threshold_mean.sort_values(by="F1-score", ascending=False)

,Accuracy,Precision,Recall,F1-score,AUC
Variable_Set,,,,,
Modelo 2 — Nível Hierárquico e Benefícios,0.762595,0.451116,0.530026,0.406405,0.8023
Modelo 8 — Integrado Multidimensional,0.773379,0.427626,0.480579,0.397574,0.7681
Modelo 6 — Perfil Pessoal e Condições de Trabalho,0.756911,0.444595,0.475768,0.372637,0.7729
Modelo 3 — Rendimento Quantitativo,0.754537,0.440579,0.480242,0.368411,0.7828
Modelo 5 — Antiguidade Organizacional,0.753432,0.429700,0.475368,0.362963,0.7542
Modelo 1 — Função Profissional Misto,0.751068,0.421184,0.466332,0.360079,0.7535
Modelo 4 — Experiência Profissional,0.744753,0.389763,0.476611,0.359626,0.7538
Modelo 7 — Reduzido Conservador Misto,0.735258,0.382179,0.442789,0.317205,0.7498


In [15]:
mix_threshold_with_f1_05 = threshold_comparison_mix[threshold_comparison_mix["F1-score"]>=0.5]

mix_best_thresholds_accuracy = mix_threshold_with_f1_05.loc[
    mix_threshold_with_f1_05.groupby(["Model"])["Accuracy"].idxmax()
].reset_index(drop=True)

mix_best_thresholds_accuracy.sort_values(by="Accuracy", ascending=False)

,Variable_Set,Model,Threshold,Accuracy,Precision,Recall,F1-score,AUC
3,Modelo 8 — Integrado Multidimensional,Logistic Regression,0.475,0.875,0.667,0.451,0.538,0.808
7,Modelo 2 — Nível Hierárquico e Benefícios,XGBoost,0.400,0.866,0.615,0.451,0.520,0.829
8,Modelo 2 — Nível Hierárquico e Benefícios,XGBoost Balanced,0.650,0.862,0.581,0.507,0.541,0.821
2,Modelo 2 — Nível Hierárquico e Benefícios,Gradient Boosting Balanced,0.625,0.857,0.562,0.507,0.533,0.824
1,Modelo 2 — Nível Hierárquico e Benefícios,Gradient Boosting,0.325,0.855,0.559,0.465,0.508,0.828
6,Modelo 2 — Nível Hierárquico e Benefícios,Random Forest Balanced,0.575,0.848,0.529,0.507,0.518,0.806
4,Modelo 8 — Integrado Multidimensional,Logistic Regression Balanced,0.650,0.844,0.511,0.634,0.566,0.810
0,Modelo 2 — Nível Hierárquico e Benefícios,Decision Tree Balanced,0.650,0.823,0.459,0.549,0.500,0.739
5,Modelo 8 — Integrado Multidimensional,Random Forest,0.200,0.789,0.408,0.690,0.513,0.787


In [16]:
mix_best_thresholds_precision = mix_threshold_with_f1_05.loc[
    mix_threshold_with_f1_05.groupby(["Model"])["Precision"].idxmax()
].reset_index(drop=True)

mix_best_thresholds_precision.sort_values(by="Precision", ascending=False)

,Variable_Set,Model,Threshold,Accuracy,Precision,Recall,F1-score,AUC
3,Modelo 8 — Integrado Multidimensional,Logistic Regression,0.525,0.873,0.674,0.408,0.509,0.808
7,Modelo 2 — Nível Hierárquico e Benefícios,XGBoost,0.400,0.866,0.615,0.451,0.520,0.829
8,Modelo 2 — Nível Hierárquico e Benefícios,XGBoost Balanced,0.650,0.862,0.581,0.507,0.541,0.821
2,Modelo 2 — Nível Hierárquico e Benefícios,Gradient Boosting Balanced,0.650,0.857,0.569,0.465,0.512,0.824
1,Modelo 2 — Nível Hierárquico e Benefícios,Gradient Boosting,0.325,0.855,0.559,0.465,0.508,0.828
6,Modelo 2 — Nível Hierárquico e Benefícios,Random Forest Balanced,0.575,0.848,0.529,0.507,0.518,0.806
4,Modelo 8 — Integrado Multidimensional,Logistic Regression Balanced,0.650,0.844,0.511,0.634,0.566,0.810
0,Modelo 2 — Nível Hierárquico e Benefícios,Decision Tree Balanced,0.650,0.823,0.459,0.549,0.500,0.739
5,Modelo 8 — Integrado Multidimensional,Random Forest,0.200,0.789,0.408,0.690,0.513,0.787


In [17]:
mix_best_thresholds_recall = mix_threshold_with_f1_05.loc[
    mix_threshold_with_f1_05.groupby(["Model"])["Recall"].idxmax()
].reset_index(drop=True)

mix_best_thresholds_recall.sort_values(by="Recall", ascending=False)

,Variable_Set,Model,Threshold,Accuracy,Precision,Recall,F1-score,AUC
4,Modelo 2 — Nível Hierárquico e Benefícios,Logistic Regression Balanced,0.475,0.753,0.372,0.775,0.502,0.829
2,Modelo 2 — Nível Hierárquico e Benefícios,Gradient Boosting Balanced,0.450,0.782,0.400,0.704,0.510,0.824
3,Modelo 2 — Nível Hierárquico e Benefícios,Logistic Regression,0.200,0.794,0.417,0.704,0.524,0.827
5,Modelo 8 — Integrado Multidimensional,Random Forest,0.200,0.789,0.408,0.690,0.513,0.787
8,Modelo 2 — Nível Hierárquico e Benefícios,XGBoost Balanced,0.450,0.796,0.417,0.676,0.516,0.821
7,Modelo 2 — Nível Hierárquico e Benefícios,XGBoost,0.200,0.803,0.425,0.634,0.508,0.829
0,Modelo 2 — Nível Hierárquico e Benefícios,Decision Tree Balanced,0.600,0.803,0.423,0.620,0.503,0.739
6,Modelo 1 — Função Profissional Misto,Random Forest Balanced,0.525,0.805,0.427,0.620,0.506,0.775
1,Modelo 2 — Nível Hierárquico e Benefícios,Gradient Boosting,0.225,0.819,0.452,0.592,0.512,0.828


In [18]:
mix_best_thresholds_auc = mix_threshold_with_f1_05.loc[
    mix_threshold_with_f1_05.groupby(["Model"])["AUC"].idxmax()
].reset_index(drop=True)

mix_best_thresholds_auc.sort_values(by="AUC", ascending=False)

,Variable_Set,Model,Threshold,Accuracy,Precision,Recall,F1-score,AUC
4,Modelo 2 — Nível Hierárquico e Benefícios,Logistic Regression Balanced,0.475,0.753,0.372,0.775,0.502,0.829
7,Modelo 2 — Nível Hierárquico e Benefícios,XGBoost,0.200,0.803,0.425,0.634,0.508,0.829
1,Modelo 2 — Nível Hierárquico e Benefícios,Gradient Boosting,0.225,0.819,0.452,0.592,0.512,0.828
3,Modelo 2 — Nível Hierárquico e Benefícios,Logistic Regression,0.200,0.794,0.417,0.704,0.524,0.827
2,Modelo 2 — Nível Hierárquico e Benefícios,Gradient Boosting Balanced,0.450,0.782,0.400,0.704,0.510,0.824
8,Modelo 2 — Nível Hierárquico e Benefícios,XGBoost Balanced,0.450,0.796,0.417,0.676,0.516,0.821
5,Modelo 2 — Nível Hierárquico e Benefícios,Random Forest,0.200,0.782,0.397,0.676,0.500,0.808
6,Modelo 2 — Nível Hierárquico e Benefícios,Random Forest Balanced,0.550,0.839,0.500,0.563,0.530,0.806
0,Modelo 2 — Nível Hierárquico e Benefícios,Decision Tree Balanced,0.600,0.803,0.423,0.620,0.503,0.739


# Comparação de Modelos Categóricos + Mistos

In [19]:
c_models_results = [general_comparison_c, threshold_comparison_c]
mix_models_results = [general_comparison_mix, threshold_comparison_mix]

for result in c_models_results:
    result["type"] = "Categorical"

for result in mix_models_results:
    result["type"] = "Mixed" 

In [20]:
all_comparison = pd.concat(
    [general_comparison_c, general_comparison_mix],
    ignore_index=True
)

all_threshold = pd.concat(
    [threshold_comparison_c, threshold_comparison_mix],
    ignore_index=True
)

In [21]:
all_comparison_by_f1 = all_threshold.loc[
    all_threshold.groupby(["Variable_Set", "Model"])["F1-score"].idxmax()
].reset_index(drop=True)

all_comparison_by_f1.sort_values(by="F1-score", ascending=False).head(20)

,Variable_Set,Model,Threshold,Accuracy,Precision,Recall,F1-score,AUC,type
144,Modelo 8 — Integrado Multidimensional,Logistic Regression,0.325,0.859,0.557,0.620,0.587,0.808,Mixed
145,Modelo 8 — Integrado Multidimensional,Logistic Regression Balanced,0.650,0.844,0.511,0.634,0.566,0.810,Mixed
38,Modelo 2 — Nível Hierárquico e Benefícios,XGBoost,0.275,0.857,0.556,0.563,0.559,0.829,Mixed
34,Modelo 2 — Nível Hierárquico e Benefícios,Logistic Regression,0.275,0.846,0.518,0.606,0.558,0.827,Mixed
28,Modelo 2 — Nível Hierárquico,XGBoost,0.250,0.834,0.489,0.634,0.552,0.808,Categorical
23,Modelo 2 — Nível Hierárquico,Gradient Boosting Balanced,0.550,0.816,0.454,0.690,0.547,0.800,Categorical
39,Modelo 2 — Nível Hierárquico e Benefícios,XGBoost Balanced,0.650,0.862,0.581,0.507,0.541,0.821,Mixed
35,Modelo 2 — Nível Hierárquico e Benefícios,Logistic Regression Balanced,0.650,0.834,0.489,0.606,0.541,0.829,Mixed
45,Modelo 3 — Faixa Salarial,Logistic Regression Balanced,0.625,0.834,0.489,0.606,0.541,0.790,Categorical
25,Modelo 2 — Nível Hierárquico,Logistic Regression Balanced,0.575,0.814,0.449,0.676,0.539,0.815,Categorical


In [22]:
all_comparison_by_model = all_threshold.loc[
    all_threshold.groupby(["Model"])["F1-score"].idxmax()
].reset_index(drop=True)

all_comparison_by_model.sort_values(by="F1-score", ascending=False)

,Variable_Set,Model,Threshold,Accuracy,Precision,Recall,F1-score,AUC,type
4,Modelo 8 — Integrado Multidimensional,Logistic Regression,0.325,0.859,0.557,0.620,0.587,0.808,Mixed
5,Modelo 8 — Integrado Multidimensional,Logistic Regression Balanced,0.650,0.844,0.511,0.634,0.566,0.810,Mixed
8,Modelo 2 — Nível Hierárquico e Benefícios,XGBoost,0.275,0.857,0.556,0.563,0.559,0.829,Mixed
3,Modelo 2 — Nível Hierárquico,Gradient Boosting Balanced,0.550,0.816,0.454,0.690,0.547,0.800,Categorical
9,Modelo 2 — Nível Hierárquico e Benefícios,XGBoost Balanced,0.650,0.862,0.581,0.507,0.541,0.821,Mixed
2,Modelo 2 — Nível Hierárquico,Gradient Boosting,0.275,0.832,0.483,0.606,0.537,0.801,Categorical
7,Modelo 2 — Nível Hierárquico e Benefícios,Random Forest Balanced,0.550,0.839,0.500,0.563,0.530,0.806,Mixed
6,Modelo 8 — Integrado Multidimensional,Random Forest,0.200,0.789,0.408,0.690,0.513,0.787,Mixed
1,Modelo 2 — Nível Hierárquico e Benefícios,Decision Tree Balanced,0.600,0.803,0.423,0.620,0.503,0.739,Mixed
0,Modelo 2 — Nível Hierárquico,Decision Tree,0.200,0.791,0.402,0.606,0.483,0.723,Categorical


In [23]:
all_comparison_threshold_mean = all_threshold.groupby("Variable_Set")[[
    "Accuracy", "Precision", "Recall", "F1-score", "AUC"
]].mean()

all_comparison_threshold_mean.sort_values(by="F1-score", ascending=False)

,Accuracy,Precision,Recall,F1-score,AUC
Variable_Set,,,,,
Modelo 2 — Nível Hierárquico e Benefícios,0.762595,0.451116,0.530026,0.406405,0.8023
Modelo 8 — Integrado Multidimensional,0.773379,0.427626,0.480579,0.397574,0.7681
Modelo 3 — Faixa Salarial,0.743742,0.438521,0.522753,0.392837,0.7698
Modelo 2 — Nível Hierárquico,0.743100,0.432711,0.530895,0.386279,0.7878
Modelo 6 — Perfil Pessoal,0.752216,0.430147,0.501716,0.380068,0.7850
Modelo 6 — Perfil Pessoal e Condições de Trabalho,0.756911,0.444595,0.475768,0.372637,0.7729
Modelo 3 — Rendimento Quantitativo,0.754537,0.440579,0.480242,0.368411,0.7828
Modelo 1 — Função Profissional,0.746863,0.420684,0.489389,0.367100,0.7644
Modelo 5 — Estabilidade e Benefícios,0.744763,0.414942,0.504042,0.363253,0.7812


In [24]:
all_threshold_with_f1_05 = all_threshold[all_threshold["F1-score"]>=0.5]

all_best_thresholds_accuracy = all_threshold_with_f1_05.loc[
    all_threshold_with_f1_05.groupby(["Model"])["Accuracy"].idxmax()
].reset_index(drop=True)

all_best_thresholds_accuracy.sort_values(by="Accuracy", ascending=False)

,Variable_Set,Model,Threshold,Accuracy,Precision,Recall,F1-score,AUC,type
3,Modelo 8 — Integrado Multidimensional,Logistic Regression,0.475,0.875,0.667,0.451,0.538,0.808,Mixed
7,Modelo 2 — Nível Hierárquico e Benefícios,XGBoost,0.400,0.866,0.615,0.451,0.520,0.829,Mixed
8,Modelo 2 — Nível Hierárquico e Benefícios,XGBoost Balanced,0.650,0.862,0.581,0.507,0.541,0.821,Mixed
2,Modelo 2 — Nível Hierárquico e Benefícios,Gradient Boosting Balanced,0.625,0.857,0.562,0.507,0.533,0.824,Mixed
1,Modelo 2 — Nível Hierárquico e Benefícios,Gradient Boosting,0.325,0.855,0.559,0.465,0.508,0.828,Mixed
6,Modelo 2 — Nível Hierárquico e Benefícios,Random Forest Balanced,0.575,0.848,0.529,0.507,0.518,0.806,Mixed
4,Modelo 5 — Estabilidade e Benefícios,Logistic Regression Balanced,0.650,0.846,0.521,0.535,0.528,0.793,Categorical
5,Modelo 2 — Nível Hierárquico,Random Forest,0.250,0.832,0.481,0.535,0.507,0.796,Categorical
0,Modelo 2 — Nível Hierárquico e Benefícios,Decision Tree Balanced,0.650,0.823,0.459,0.549,0.500,0.739,Mixed


In [25]:
all_best_thresholds_precision = all_threshold_with_f1_05.loc[
    all_threshold_with_f1_05.groupby(["Model"])["Precision"].idxmax()
].reset_index(drop=True)

all_best_thresholds_precision.sort_values(by="Precision", ascending=False)

,Variable_Set,Model,Threshold,Accuracy,Precision,Recall,F1-score,AUC,type
3,Modelo 8 — Integrado Multidimensional,Logistic Regression,0.525,0.873,0.674,0.408,0.509,0.808,Mixed
7,Modelo 2 — Nível Hierárquico e Benefícios,XGBoost,0.400,0.866,0.615,0.451,0.520,0.829,Mixed
8,Modelo 2 — Nível Hierárquico e Benefícios,XGBoost Balanced,0.650,0.862,0.581,0.507,0.541,0.821,Mixed
2,Modelo 2 — Nível Hierárquico e Benefícios,Gradient Boosting Balanced,0.650,0.857,0.569,0.465,0.512,0.824,Mixed
1,Modelo 2 — Nível Hierárquico e Benefícios,Gradient Boosting,0.325,0.855,0.559,0.465,0.508,0.828,Mixed
6,Modelo 2 — Nível Hierárquico e Benefícios,Random Forest Balanced,0.575,0.848,0.529,0.507,0.518,0.806,Mixed
4,Modelo 5 — Estabilidade e Benefícios,Logistic Regression Balanced,0.650,0.846,0.521,0.535,0.528,0.793,Categorical
5,Modelo 2 — Nível Hierárquico,Random Forest,0.250,0.832,0.481,0.535,0.507,0.796,Categorical
0,Modelo 2 — Nível Hierárquico e Benefícios,Decision Tree Balanced,0.650,0.823,0.459,0.549,0.500,0.739,Mixed


In [26]:
all_best_thresholds_recall = all_threshold_with_f1_05.loc[
    all_threshold_with_f1_05.groupby(["Model"])["Recall"].idxmax()
].reset_index(drop=True)

all_best_thresholds_recall.sort_values(by="Recall", ascending=False)

,Variable_Set,Model,Threshold,Accuracy,Precision,Recall,F1-score,AUC,type
4,Modelo 2 — Nível Hierárquico e Benefícios,Logistic Regression Balanced,0.475,0.753,0.372,0.775,0.502,0.829,Mixed
2,Modelo 2 — Nível Hierárquico,Gradient Boosting Balanced,0.475,0.762,0.379,0.746,0.502,0.800,Categorical
8,Modelo 2 — Nível Hierárquico,XGBoost Balanced,0.475,0.787,0.411,0.746,0.530,0.804,Categorical
3,Modelo 2 — Nível Hierárquico e Benefícios,Logistic Regression,0.200,0.794,0.417,0.704,0.524,0.827,Mixed
1,Modelo 2 — Nível Hierárquico,Gradient Boosting,0.200,0.796,0.419,0.690,0.521,0.801,Categorical
5,Modelo 8 — Integrado Multidimensional,Random Forest,0.200,0.789,0.408,0.690,0.513,0.787,Mixed
7,Modelo 2 — Nível Hierárquico,XGBoost,0.200,0.798,0.418,0.648,0.508,0.808,Categorical
0,Modelo 2 — Nível Hierárquico e Benefícios,Decision Tree Balanced,0.600,0.803,0.423,0.620,0.503,0.739,Mixed
6,Modelo 1 — Função Profissional Misto,Random Forest Balanced,0.525,0.805,0.427,0.620,0.506,0.775,Mixed


In [27]:
all_best_thresholds_auc = all_threshold_with_f1_05.loc[
    all_threshold_with_f1_05.groupby(["Model"])["AUC"].idxmax()
].reset_index(drop=True)

all_best_thresholds_auc.sort_values(by="AUC", ascending=False)

,Variable_Set,Model,Threshold,Accuracy,Precision,Recall,F1-score,AUC,type
4,Modelo 2 — Nível Hierárquico e Benefícios,Logistic Regression Balanced,0.475,0.753,0.372,0.775,0.502,0.829,Mixed
7,Modelo 2 — Nível Hierárquico e Benefícios,XGBoost,0.200,0.803,0.425,0.634,0.508,0.829,Mixed
1,Modelo 2 — Nível Hierárquico e Benefícios,Gradient Boosting,0.225,0.819,0.452,0.592,0.512,0.828,Mixed
3,Modelo 2 — Nível Hierárquico e Benefícios,Logistic Regression,0.200,0.794,0.417,0.704,0.524,0.827,Mixed
2,Modelo 2 — Nível Hierárquico e Benefícios,Gradient Boosting Balanced,0.450,0.782,0.400,0.704,0.510,0.824,Mixed
8,Modelo 2 — Nível Hierárquico e Benefícios,XGBoost Balanced,0.450,0.796,0.417,0.676,0.516,0.821,Mixed
5,Modelo 2 — Nível Hierárquico e Benefícios,Random Forest,0.200,0.782,0.397,0.676,0.500,0.808,Mixed
6,Modelo 2 — Nível Hierárquico e Benefícios,Random Forest Balanced,0.550,0.839,0.500,0.563,0.530,0.806,Mixed
0,Modelo 2 — Nível Hierárquico e Benefícios,Decision Tree Balanced,0.600,0.803,0.423,0.620,0.503,0.739,Mixed


# Cross-Validation

In [28]:
cv_comparison = run_cross_validation_mixed(df=df, models_dict=all_models_dict_c, estimators_dict=estimators_dict, target="AttritionFlag")

cv_comparison.sort_values("F1_Mean", ascending=False)

,Variable_Set,Model,Accuracy_Mean,Accuracy_Std,Precision_Mean,Precision_Std,Recall_Mean,Recall_Std,F1_Mean,F1_Std,AUC_Mean,AUC_Std,N_Numeric_Variables,N_Categorical_Variables,N_Features_After_Dummies
141,Modelo 8 — Integrado Multidimensional,Logistic Regression Balanced,0.776,0.035,0.401,0.051,0.776,0.085,0.528,0.062,0.839,0.055,7,11,43
147,Modelo 8 — Integrado Multidimensional,Gradient Boosting Balanced,0.818,0.018,0.454,0.048,0.595,0.086,0.512,0.049,0.805,0.047,7,11,43
149,Modelo 8 — Integrado Multidimensional,XGBoost Balanced,0.814,0.028,0.442,0.064,0.603,0.099,0.510,0.076,0.808,0.051,7,11,43
81,Modelo 2 — Nível Hierárquico e Benefícios,Logistic Regression Balanced,0.754,0.030,0.371,0.039,0.737,0.079,0.492,0.048,0.814,0.047,3,8,25
145,Modelo 8 — Integrado Multidimensional,Random Forest Balanced,0.797,0.034,0.418,0.069,0.599,0.079,0.490,0.063,0.803,0.039,7,11,43
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64,Modelo 7 — Reduzido Conservador,Random Forest,0.840,0.003,0.200,0.400,0.008,0.017,0.016,0.032,0.728,0.051,0,7,18
14,Modelo 2 — Nível Hierárquico,Random Forest,0.839,0.004,0.100,0.300,0.004,0.012,0.008,0.024,0.758,0.049,0,8,22
4,Modelo 1 — Função Profissional,Random Forest,0.839,0.005,0.100,0.300,0.004,0.013,0.008,0.025,0.768,0.044,0,8,26
134,Modelo 7 — Reduzido Conservador Misto,Random Forest,0.839,0.003,0.000,0.000,0.000,0.000,0.000,0.000,0.729,0.056,3,6,18


In [29]:
best_cv_result = cv_comparison.loc[
    cv_comparison.groupby(["Variable_Set", "Model"])["F1_Mean"].idxmax()
].reset_index(drop=True)

best_cv_result.sort_values(by="F1_Mean", ascending=False).head(20)

,Variable_Set,Model,Accuracy_Mean,Accuracy_Std,Precision_Mean,Precision_Std,Recall_Mean,Recall_Std,F1_Mean,F1_Std,AUC_Mean,AUC_Std,N_Numeric_Variables,N_Categorical_Variables,N_Features_After_Dummies
145,Modelo 8 — Integrado Multidimensional,Logistic Regression Balanced,0.776,0.035,0.401,0.051,0.776,0.085,0.528,0.062,0.839,0.055,7,11,43
143,Modelo 8 — Integrado Multidimensional,Gradient Boosting Balanced,0.818,0.018,0.454,0.048,0.595,0.086,0.512,0.049,0.805,0.047,7,11,43
149,Modelo 8 — Integrado Multidimensional,XGBoost Balanced,0.814,0.028,0.442,0.064,0.603,0.099,0.510,0.076,0.808,0.051,7,11,43
35,Modelo 2 — Nível Hierárquico e Benefícios,Logistic Regression Balanced,0.754,0.030,0.371,0.039,0.737,0.079,0.492,0.048,0.814,0.047,3,8,25
147,Modelo 8 — Integrado Multidimensional,Random Forest Balanced,0.797,0.034,0.418,0.069,0.599,0.079,0.490,0.063,0.803,0.039,7,11,43
148,Modelo 8 — Integrado Multidimensional,XGBoost,0.876,0.021,0.735,0.141,0.367,0.070,0.486,0.087,0.814,0.053,7,11,43
144,Modelo 8 — Integrado Multidimensional,Logistic Regression,0.872,0.022,0.698,0.126,0.372,0.095,0.480,0.101,0.844,0.051,7,11,43
65,Modelo 4 — Experiência Profissional,Logistic Regression Balanced,0.737,0.045,0.356,0.048,0.746,0.078,0.480,0.054,0.787,0.051,4,6,19
73,Modelo 4 — Trajetória Organizacional,Gradient Boosting Balanced,0.762,0.045,0.376,0.069,0.662,0.055,0.477,0.063,0.771,0.039,0,8,22
37,Modelo 2 — Nível Hierárquico e Benefícios,Random Forest Balanced,0.780,0.029,0.388,0.050,0.615,0.072,0.474,0.051,0.774,0.044,3,8,25


In [30]:
top_combinations = (
    cv_comparison
    .sort_values("F1_Mean", ascending=False)
    .head(20)[["Variable_Set", "Model"]]
    .to_dict("records")
)

tuning_results_df, best_models = tune_hyperparameters_top_combinations(
    df=df,
    models_dict=all_models_dict_c,
    estimators_dict=estimators_dict,
    param_distributions_dict=param_distributions_dict,
    top_combinations=top_combinations,
    target="AttritionFlag",
    n_iter=30,
    n_splits=10,
    scoring="f1"
)

tuning_results_df.sort_values("Best_Score", ascending=False)

/Users/nathansperinde/Desktop/Portifolio/projeto_1/.venv/lib/python3.13/site-packages/sklearn/model_selection/_search.py:326: UserWarning: The total space of parameters 10 is smaller than n_iter=30. Running 10 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(
/Users/nathansperinde/Desktop/Portifolio/projeto_1/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/nathansperinde/Desktop/Portifolio/projeto_1/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is depr

,Variable_Set,Model,Best_Score,Scoring,Best_Params,N_Numeric_Variables,N_Categorical_Variables,N_Features_After_Dummies
0,Modelo 8 — Integrado Multidimensional,Logistic Regression Balanced,0.532,f1,"{'classifier__solver': 'liblinear', 'classifie...",7,11,43
1,Modelo 8 — Integrado Multidimensional,Gradient Boosting Balanced,0.519,f1,"{'classifier__subsample': 0.7, 'classifier__n_...",7,11,43
2,Modelo 8 — Integrado Multidimensional,XGBoost Balanced,0.512,f1,"{'classifier__subsample': 0.9, 'classifier__sc...",7,11,43
4,Modelo 8 — Integrado Multidimensional,Random Forest Balanced,0.511,f1,"{'classifier__n_estimators': 200, 'classifier_...",7,11,43
7,Modelo 8 — Integrado Multidimensional,Logistic Regression,0.499,f1,"{'classifier__solver': 'liblinear', 'classifie...",7,11,43
5,Modelo 8 — Integrado Multidimensional,XGBoost,0.491,f1,"{'classifier__subsample': 1.0, 'classifier__re...",7,11,43
3,Modelo 2 — Nível Hierárquico e Benefícios,Logistic Regression Balanced,0.490,f1,"{'classifier__solver': 'liblinear', 'classifie...",3,8,25
9,Modelo 2 — Nível Hierárquico e Benefícios,Random Forest Balanced,0.486,f1,"{'classifier__n_estimators': 100, 'classifier_...",3,8,25
10,Modelo 6 — Perfil Pessoal,Logistic Regression Balanced,0.480,f1,"{'classifier__solver': 'liblinear', 'classifie...",0,9,24
6,Modelo 4 — Experiência Profissional,Logistic Regression Balanced,0.480,f1,"{'classifier__solver': 'liblinear', 'classifie...",4,6,19


In [31]:
base_top_results = (
    cv_comparison
    .sort_values("F1_Mean", ascending=False)
    .head(20)
    [["Variable_Set", "Model", "F1_Mean", "F1_Std", "AUC_Mean"]]
)

optimized_results = tuning_results_df[["Variable_Set", "Model", "Best_Score", "Best_Params"]]

comparison_base_vs_optimized = base_top_results.merge(optimized_results, on=["Variable_Set", "Model"], how="left")

comparison_base_vs_optimized["F1_Improvement"] = (comparison_base_vs_optimized["Best_Score"] - comparison_base_vs_optimized["F1_Mean"])

comparison_base_vs_optimized.sort_values(by="Best_Score", ascending=False)

,Variable_Set,Model,F1_Mean,F1_Std,AUC_Mean,Best_Score,Best_Params,F1_Improvement
0,Modelo 8 — Integrado Multidimensional,Logistic Regression Balanced,0.528,0.062,0.839,0.532,"{'classifier__solver': 'liblinear', 'classifie...",0.004
1,Modelo 8 — Integrado Multidimensional,Gradient Boosting Balanced,0.512,0.049,0.805,0.519,"{'classifier__subsample': 0.7, 'classifier__n_...",0.007
2,Modelo 8 — Integrado Multidimensional,XGBoost Balanced,0.510,0.076,0.808,0.512,"{'classifier__subsample': 0.9, 'classifier__sc...",0.002
4,Modelo 8 — Integrado Multidimensional,Random Forest Balanced,0.490,0.063,0.803,0.511,"{'classifier__n_estimators': 200, 'classifier_...",0.021
7,Modelo 8 — Integrado Multidimensional,Logistic Regression,0.480,0.101,0.844,0.499,"{'classifier__solver': 'liblinear', 'classifie...",0.019
5,Modelo 8 — Integrado Multidimensional,XGBoost,0.486,0.087,0.814,0.491,"{'classifier__subsample': 1.0, 'classifier__re...",0.005
3,Modelo 2 — Nível Hierárquico e Benefícios,Logistic Regression Balanced,0.492,0.048,0.814,0.490,"{'classifier__solver': 'liblinear', 'classifie...",-0.002
9,Modelo 2 — Nível Hierárquico e Benefícios,Random Forest Balanced,0.474,0.051,0.774,0.486,"{'classifier__n_estimators': 100, 'classifier_...",0.012
10,Modelo 6 — Perfil Pessoal,Logistic Regression Balanced,0.473,0.038,0.793,0.480,"{'classifier__solver': 'liblinear', 'classifie...",0.007
6,Modelo 4 — Experiência Profissional,Logistic Regression Balanced,0.480,0.054,0.787,0.480,"{'classifier__solver': 'liblinear', 'classifie...",0.000


In [32]:
top_20_best_models = {
    (row["Variable_Set"], row["Model"]): best_models[(row["Variable_Set"], row["Model"])]
    for _, row in (
        tuning_results_df
        .sort_values("Best_Score", ascending=False)
        .iterrows()
    )
}

threshold_comparison_opt, best_thresholds_opt, confusion_results_opt, fitted_models_opt = evaluate_thresholds_optimized_models(
    df=df,
    models_dict=all_models_dict_c,
    best_models=top_20_best_models,
    estimators_dict=estimators_dict,
    target="AttritionFlag"
)

best_thresholds_opt

/Users/nathansperinde/Desktop/Portifolio/projeto_1/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/nathansperinde/Desktop/Portifolio/projeto_1/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penal

,Variable_Set,Model,Threshold,Accuracy,Precision,Recall,F1-score,AUC
15,Modelo 8 — Integrado Multidimensional,Logistic Regression,0.38,0.866,0.583,0.592,0.587,0.814
1,Modelo 2 — Nível Hierárquico e Benefícios,Logistic Regression Balanced,0.70,0.864,0.582,0.549,0.565,0.830
16,Modelo 8 — Integrado Multidimensional,Logistic Regression Balanced,0.66,0.841,0.506,0.620,0.557,0.810
0,Modelo 2 — Nível Hierárquico,Logistic Regression Balanced,0.59,0.823,0.466,0.676,0.552,0.817
13,Modelo 6 — Perfil Pessoal e Condições de Trabalho,Logistic Regression Balanced,0.65,0.841,0.506,0.563,0.533,0.802
12,Modelo 6 — Perfil Pessoal,Logistic Regression Balanced,0.63,0.830,0.477,0.592,0.528,0.802
2,Modelo 2 — Nível Hierárquico e Benefícios,Random Forest Balanced,0.54,0.853,0.545,0.507,0.526,0.807
11,Modelo 5 — Estabilidade e Benefícios,Logistic Regression Balanced,0.64,0.839,0.500,0.549,0.523,0.795
5,Modelo 4 — Experiência Profissional,Logistic Regression Balanced,0.61,0.819,0.451,0.577,0.506,0.795
3,Modelo 3 — Rendimento Quantitativo,Random Forest Balanced,0.48,0.796,0.414,0.648,0.505,0.782
